# 21 — Coming from PySpark: Migration Guide

**Shared perspective.** Side-by-side mapping of everyday PySpark idioms to IrisPark. If a cell runs here, the same shape works in your existing Spark code with a one-line import change — and where behavior differs, notebook 10 has the details.

**The one import rule**: everything comes from `irispark` instead of `pyspark.sql`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Session & imports

```python
# PySpark                                   # IrisPark
from pyspark.sql import SparkSession        from irispark import IrisParkSession
spark = SparkSession.builder.appName(...).getOrCreate()
                                            session = IrisParkSession.builder().host(...).getOrCreate()
from pyspark.sql.functions import col, lit  from irispark.functions import col, lit
```
Same builder chain, same lazy DataFrame, same function names.

In [ ]:
from irispark.functions import col, lit, when
print(type(col("x")).__name__, type(lit(1)).__name__)

## 2. Creating DataFrames

`createDataFrame` accepts pandas DataFrames, list-of-dicts and row tuples — like Spark's local-mode entry points.

In [ ]:
import pandas as pd

pdf = pd.DataFrame({"nome": ["Ana", "Bruno"], "valor": [100.0, 200.0]})
df = session.createDataFrame(pdf)
df.show()

## 3. Core transformations — identical signatures

`select`, `filter`/`where`, `withColumn`, `groupBy().agg()`, `orderBy` all match PySpark shapes.

In [ ]:
from irispark.functions import sum as s, avg

(df.filter("valor > 50")
   .withColumn("dobro", col("valor") * 2)
   .groupBy("nome")
   .agg(s("valor").alias("total"), avg("valor").alias("media"))
   .orderBy("total DESC")
   .show())

## 4. Actions & interop

`count/head/take/first` behave the same; `toPandas` becomes `to_pandas` (both names work).

In [ ]:
print("count:", df.count(), "| first:", df.first()["nome"])
pdf_out = df.to_pandas()
print("type:", type(pdf_out).__name__)

## 5. na namespace & writers

Same method names; writer modes match (`error` default included).

In [ ]:
df.na.fill("?").show()

# df.write.mode("append").saveAsTable("t")  # identical shape
print("writer modes: error | append | overwrite | ignore | errorifexists")

## 6. What to check before migrating

1. **Known differences** (notebook 10 / `docs/known_differences.md`): float rounding edge cases, `pow` domain errors → NULL, `regexp_extract` no-match → NULL, `stat.cov` empty → None.
2. **Unsupported constructs**: arrays/maps/structs, JSON family, `groupingSets`, bitwise ops.
3. **Performance model**: no executors — every operation is IRIS SQL pushdown; design filters to stay pushable (`docs/performance_guide.md`).

In [ ]:
print("migration checklist: known differences -> unsupported APIs -> pushdown-friendly filters")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")